In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

carpeta = Path("resultados/cierre_managua").resolve()
ruta_modelo = carpeta / "ridge_managua_v1.joblib"
ruta_celdas = carpeta / "celdas_e_historial_v1.csv"

for ruta in (ruta_modelo, ruta_celdas):
    if not ruta.is_file():
        raise FileNotFoundError(f"No encontré este archivo: {ruta}")

paquete = joblib.load(ruta_modelo)
celdas = pd.read_csv(ruta_celdas, dtype={"id_celda": "string"})

fecha = pd.Timestamp("2027-02-15")
celdas["dia_del_anio"] = fecha.dayofyear

prediccion = paquete["modelo"].predict(celdas[paquete["variables"]])

resultado = celdas[["id_celda", "col_celda", "fila_celda", "n_previas"]].copy()
resultado["fecha_objetivo"] = fecha.date().isoformat()
resultado["temperatura_estimada_c"] = prediccion
resultado["temperatura_historial_c"] = celdas["media_previa_c"]
resultado["version_modelo"] = paquete["version"]

if resultado.empty or not resultado["id_celda"].is_unique:
    raise ValueError("Esperaba una predicción por cada celda.")

salida = carpeta / "prueba_kernel_limpio"
salida.mkdir(exist_ok=True)
resultado.to_csv(salida / "prediccion_demostracion.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 7))
mapa = ax.scatter(
    (resultado["col_celda"] + 0.5) * 250,
    (resultado["fila_celda"] + 0.5) * 250,
    c=resultado["temperatura_estimada_c"],
    cmap="inferno",
    marker="s",
    s=7,
    linewidths=0,
)
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title(f"Managua · Temperatura superficial estimada\n{fecha:%d/%m/%Y} · Demostración")
fig.colorbar(mapa, ax=ax, label="Temperatura superficial estimada (°C)")
fig.savefig(salida / "mapa_demostracion.png", dpi=180, bbox_inches="tight")
plt.show()

print("Modelo:", paquete["version"])
print(f"Celdas predichas: {len(resultado):,}")
print("Archivos generados:", salida)

In [ ]:
from pathlib import Path
import shutil

raiz = Path.cwd()

# Si Positron está abierto en la carpeta padre, entra al proyecto.
if (raiz / "calor_superficial" / "data").is_dir():
    raiz = raiz / "calor_superficial"

archivos_requeridos = [
    Path("data/predictores/managua_panel_con_predictores.csv"),
    Path("resultados/cierre_managua/ridge_managua_v1.joblib"),
    Path("resultados/cierre_managua/celdas_e_historial_v1.csv"),
]

faltantes = [ruta for ruta in archivos_requeridos if not (raiz / ruta).is_file()]
if faltantes:
    detalle = "\n".join(str(raiz / ruta) for ruta in faltantes)
    raise FileNotFoundError(f"No encontré estos archivos:\n{detalle}")

# Usar una carpeta nueva en cada ejecución mantiene las copias anteriores.
entrega = raiz / "entrega_calor_managua"
numero = 2
while entrega.exists() or entrega.with_suffix(".zip").exists():
    entrega = raiz / f"entrega_calor_managua_{numero}"
    numero += 1

entrega.mkdir()

# Mantener las rutas originales para poder ejecutar los notebooks desde el ZIP.
panel = archivos_requeridos[0]
(entrega / panel.parent).mkdir(parents=True)
shutil.copy2(raiz / panel, entrega / panel)

cierre = Path("resultados/cierre_managua")
shutil.copytree(
    raiz / cierre,
    entrega / cierre,
    ignore=shutil.ignore_patterns(".ipynb_checkpoints", "__pycache__"),
)

# Copiar figuras y tablas descriptivas sin incluir los GeoTIFF originales.
figuras = [
    Path("data/celdas_por_escena_2023_2026/mapa_recurrencia_15_fechas.png"),
    Path("data/celdas_por_escena_2023_2026/recurrencia_multianual_15_fechas.csv"),
    Path("data/piloto/managua_recurrencia_cinco_fechas_2026.png"),
]
for ruta in figuras:
    if (raiz / ruta).is_file():
        (entrega / ruta.parent).mkdir(parents=True, exist_ok=True)
        shutil.copy2(raiz / ruta, entrega / ruta)

# Guardar los resultados que todavía estén en memoria.
tablas = {
    "test_temp": "test_temp_2026.csv",
    "metricas_temp": "metricas_temperatura_2026.csv",
    "metricas_meteo": "metricas_temperatura_aire_2026.csv",
    "metricas_regla_2026": "metricas_radiacion_2026.csv",
    "ablacion_por_fecha": "ablacion_clasificacion_2026.csv",
}

for nombre, archivo in tablas.items():
    if nombre in globals():
        globals()[nombre].to_csv(entrega / "resultados" / archivo, index=False)
        print("Tabla guardada:", archivo)

if "test_temp" not in globals():
    print("test_temp no está en memoria. Puede reconstruirse con el panel y el notebook.")
elif not {"pred_ngboost_c", "limite_inferior_90_c", "limite_superior_90_c"}.issubset(
    test_temp.columns
):
    print("Se guardó test_temp. Los intervalos de NGBoost quedan por reconstruir.")

ruta_zip = shutil.make_archive(
    str(entrega), "zip", root_dir=entrega.parent, base_dir=entrega.name
)
print("\nCarpeta:", entrega.resolve())
print("ZIP para adjuntar:", ruta_zip)